# Séparer les environnements de vecteurs

> **La séparation par environnement est un contrôle d'accès, pas une commodité d'organisation.**
> Corollaire opérationnel : une réindexation lancée sans préciser l'environnement cible écrase un corpus voisin.

Ces deux lignes viennent du *Parcours 2* de `livresagites-parcours.md`. Elles décrivent deux défaillances observées sur une installation réelle d'AI-Engine, où les vecteurs ne forment pas un corpus unique mais **six environnements d'embeddings distincts**, séparés par un identifiant en base.

Ce notebook ne reparle pas de l'observation : il **rend les deux défaillances mesurables et reproductibles**. À partir d'un vector store synthétique partitionné en six régimes d'accès, on mesure deux choses :

1. **le taux de fuite** — une requête de retrieval émise *sans* filtre d'environnement renvoie des chunks d'un régime d'accès différent ;
2. **l'accident de réindexation** — `reindexer(..., environnement=None)` écrit dans un emplacement par défaut qui écrase un corpus voisin, silencieusement.

Les deux sont démontrés déterministiquement, sans réseau, sans clé, sur une fixture synthétique.

## La scène : Maison Valmont et ses six régimes d'accès

Maison Valmont indexe ses contenus dans un vector store, mais tous les
contenus ne s'adressent pas au même public. Six environnements cohabitent,
chacun destiné à un régime d'accès distinct :

| Environnement | Public visé | Régime |
|---|---|---|
| `catalogue_public` | visiteurs anonymes | ouvert |
| `vitrine` | page d'accueil, mis en avant | ouvert |
| `comite_lecture` | comité interne de relecture | réservé |
| `atelier_interne` | notes d'atelier, brouillons | réservé |
| `archive_privee` | archives dirigeants | confidentiel |
| `logistique` | fiches fournisseurs, stocks | interne |

Deux environnements — `catalogue_public` et `comite_lecture` — traitent des
**œuvres comparables** (mêmes pièces, mêmes collections). Géométriquement,
leurs vecteurs sont donc proches : c'est précisément ce chevauchement qui
rendra la fuite possible, et qui montre pourquoi le contrôle d'accès ne
peut pas se déduire de la seule distance vectorielle.

In [1]:
# Dépendances : numpy uniquement. Aucun réseau, aucune clé, aucun modèle.
import numpy as np

RNG = np.random.default_rng(7)   # seed fixée : tout le notebook est reproductible
DIM = 16                         # dimension de l'espace d'embedding (synthétique)
ENVIRONNEMENTS = [
    "catalogue_public",
    "comite_lecture",
    "atelier_interne",
    "logistique",
    "archive_privee",
    "vitrine",
]

## Construire le vector store partitionné

Chaque environnement est un **cluster** : un centroïde tiré aléatoirement,
autour duquel s'organisent `N=40` chunks. Le détail important : on
rapproche délibérément `catalogue_public` et `comite_lecture` (ils partagent
un thème, comme dans la vraie vie), de sorte que leurs clusters se
chevauchent partiellement. Les quatre autres environnements restent bien
séparés.

In [2]:
N_PAR_ENV = 40

# centroïdes de base, un par environnement
centroids = {env: RNG.normal(scale=3.0, size=DIM) for env in ENVIRONNEMENTS}

# rapprochement catalogue_public <-> comite_lecture (mêmes œuvres => vecteurs proches)
centroids["comite_lecture"] = centroids["catalogue_public"] + RNG.normal(scale=0.6, size=DIM)

# le vector store : dictionnaire environnement -> (N, DIM) chunks
store = {}
for env in ENVIRONNEMENTS:
    store[env] = centroids[env] + RNG.normal(scale=1.5, size=(N_PAR_ENV, DIM))

print(f"Vector store Maison Valmont : {len(store)} environnements, "
      f"{sum(len(v) for v in store.values())} chunks au total, dimension {DIM}.")
for env in ENVIRONNEMENTS:
    print(f"  {env:18s} : {len(store[env]):2d} vecteurs")

Vector store Maison Valmont : 6 environnements, 240 chunks au total, dimension 16.
  catalogue_public   : 40 vecteurs
  comite_lecture     : 40 vecteurs
  atelier_interne    : 40 vecteurs
  logistique         : 40 vecteurs
  archive_privee     : 40 vecteurs
  vitrine            : 40 vecteurs


## Le geste attendu : filtrer par environnement

Un retrieval correct, émis depuis un chatbot pointant vers
`catalogue_public`, ne doit chercher **que** parmi les chunks de
`catalogue_public`. C'est ce que fait le paramètre `filtre` : il restreint
la recherche à un environnement avant le top-k. Sans lui, la recherche
se fait sur **l'union** de tous les environnements.

In [3]:
def retrieve(query, store, k=5, filtre=None):
    """Renvoie les k chunks les plus proches de query (distance L2).

    Si filtre=None : cherche dans TOUS les environnements (dangereux).
    Sinon : ne cherche que dans l'environnement 'filtre' (contrôle d'accès).
    """
    envs_cherches = [filtre] if filtre is not None else list(store.keys())
    vecs, labels = [], []
    for env in envs_cherches:
        for v in store[env]:
            vecs.append(v); labels.append(env)
    vecs = np.array(vecs)
    dists = np.linalg.norm(vecs - query, axis=1)
    order = np.argsort(dists)[:k]
    return [labels[j] for j in order]

def taux_de_fuite(resultats, environnement_attendu):
    """Fraction des resultats hors de l'environnement attendu."""
    if not resultats:
        return 0.0
    hors = sum(1 for r in resultats if r != environnement_attendu)
    return hors / len(resultats)

## Première défaillance — la fuite cross-environnement

Un visiteur **public** pose une question. Sa requête est embedée, puis on
cherche les 5 chunks les plus proches. Si l'environnement cible n'est pas
précisé, le retrieve balaie tout le store.

In [4]:
# Une requête issue du regime catalogue_public (un visiteur cherche une œuvre)
requete = store["catalogue_public"][0]

# retrieve SANS filtre : la recherche balaie les six environnements
top5_sans_filtre = retrieve(requete, store, k=5, filtre=None)
fuite = taux_de_fuite(top5_sans_filtre, "catalogue_public")

print("Top-5 sans filtre :", top5_sans_filtre)
print(f"Taux de fuite : {fuite:.0%} des chunks renvoyés viennent d'un autre régime.")
if fuite > 0:
    intrus = [r for r in top5_sans_filtre if r != "catalogue_public"]
    print(f"  -> un visiteur public reçoit du contenu de : {intrus}")

Top-5 sans filtre : ['catalogue_public', 'catalogue_public', 'catalogue_public', 'catalogue_public', 'comite_lecture']
Taux de fuite : 20% des chunks renvoyés viennent d'un autre régime.
  -> un visiteur public reçoit du contenu de : ['comite_lecture']


### Lecture

Le taux de fuite n'est pas un artefact de géométrie trop lâche : il vient
de ce que `catalogue_public` et `comite_lecture` parlent des **même œuvres**.
Leurs vecteurs sont légitimement proches, donc le plus proche voisin d'un
chunk public peut être un chunk du comité. La distance vectorielle, à elle
seule, **ne sait rien** du régime d'accès — elle ne mesure que la
proximité sémantique. C'est exactement le cas où un filtre explicite est
indispensable.

In [5]:
# Même requête, cette fois AVEC le filtre attendu
top5_avec_filtre = retrieve(requete, store, k=5, filtre="catalogue_public")
fuite_f = taux_de_fuite(top5_avec_filtre, "catalogue_public")

print("Top-5 avec filtre 'catalogue_public' :", top5_avec_filtre)
print(f"Taux de fuite : {fuite_f:.0%}")
print()
print("--- Comparaison ---")
print(f"  sans filtre : {fuite:.0%} de fuite (contenu réservé livré au public)")
print(f"  avec filtre : {fuite_f:.0%} de fuite (recherche scopée au régime attendu)")

Top-5 avec filtre 'catalogue_public' : ['catalogue_public', 'catalogue_public', 'catalogue_public', 'catalogue_public', 'catalogue_public']
Taux de fuite : 0%

--- Comparaison ---
  sans filtre : 20% de fuite (contenu réservé livré au public)
  avec filtre : 0% de fuite (recherche scopée au régime attendu)


### Lecture

Le filtre ramène le taux de fuite à **0%** — non pas en rapprochant les
vecteurs, mais en **restreignant l'espace de recherche avant** le top-k.
C'est bien un contrôle d'accès : la question n'est pas « ce chunk est-il
sémantiquement pertinent ? » mais « ce visiteur a-t-il le droit de le
voir ? ». La distance vectorielle répond à la première ; seule la
partition par environnement répond à la seconde.

## Deuxième défaillance — l'accident de réindexation

On veut réindexer un nouveau corpus. La fonction `reindexer` prend un
environnement cible. Si l'oubli de cet argument fait tomber sur un
emplacement par défaut, cet emplacement **écrase** un environnement
existant — sans avertissement.

In [6]:
def reindexer(nouveau_corpus, store, environnement=None):
    """Réindexe nouveau_corpus dans l'environnement cible.

    DANGER : si environnement=None, écrit dans le premier environnement
    du store (emplacement par défaut) et écrase son contenu.
    """
    cible = environnement if environnement is not None else next(iter(store))
    store[cible] = nouveau_corpus
    return cible

# état avant : catalogue_public contient ses 40 chunks d'origine
print("AVANT accident :")
print(f"  catalogue_public : {len(store['catalogue_public'])} vecteurs")
print(f"  comite_lecture   : {len(store['comite_lecture'])} vecteurs")

# on voulait réindexer la vitrine, mais on a oublié l'environnement cible
nouveau_corpus_vitrine = RNG.normal(scale=3.0, size=(12, DIM))
cible_touchée = reindexer(nouveau_corpus_vitrine, store, environnement=None)
print(f"\nreindexer(..., environnement=None) a écrit dans : {cible_touchée}")

print("\nAPRÈS accident :")
for env in ["catalogue_public", "comite_lecture"]:
    print(f"  {env:18s} : {len(store[env])} vecteurs")

AVANT accident :
  catalogue_public : 40 vecteurs
  comite_lecture   : 40 vecteurs

reindexer(..., environnement=None) a écrit dans : catalogue_public

APRÈS accident :
  catalogue_public   : 12 vecteurs
  comite_lecture     : 40 vecteurs


### Lecture

L'environnement par défaut était `catalogue_public`. En voulant réindexer
la vitrine, on a **silencieusement détruit** les 40 chunks d'origine du
catalogue public, remplacés par les 12 nouveaux vecteurs. Aucun message,
aucune erreur — la fonction a fait exactement ce qu'on lui a dit. La
perte est immédiate et invisible tant qu'on ne compte pas.

C'est le corollaire opérationnel du parcours : « une réindexation lancée
sans préciser l'environnement cible écrase un corpus voisin ». Le comptage
avant/après est le seul instrument qui révèle l'accident.

## La leçon, mesurée

Les deux défaillances ont la même racine : **la partition par environnement
est porteuse d'un invariant de sécurité, pas d'une commodité de rangement**.
Retirer le filtre ou oublier la cible ne dégrade pas la qualité du
retrieval — il viole un contrôle d'accès.

Deux métriques suffisent à le surveiller :

- le **taux de fuite** d'un retrieval (fraction du top-k hors environnement
  attendu) — il doit être `0%` en production ;
- le **compte de chunks** par environnement avant/après une réindexation —
  toute chute inattendue signale un écrasement.

Sur l'installation observée, ces deux métriques n'étaient instrumentées
nulle part : la fuite était invisible (le chatbot répondait, c'est tout),
et l'écrasement silencieux (le store acceptait l'écriture). Les rendre
explicites est le premier pas pour qu'elles cessent d'arriver.

## Exercices

Les trois exercices suivent le notebook : mesurer, protéger, surveiller.
Aucun n'est corrigé — à toi de compléter le geste à partir des fonctions
déjà définies (`retrieve`, `taux_de_fuite`, `reindexer`).

### Exercice 1 — L'effet du top-k sur la fuite

Le taux de fuite dépend de `k` : plus on demande de chunks, plus on a de
chances d'attraper un voisin hors-régime. Calcule le taux de fuite **moyen**
d'une requête `catalogue_public` sans filtre, pour `k = 5`, puis `k = 10`,
puis `k = 20`. Observe-t-on une dégradation ?

*Indice :* boucle sur une dizaine de requêtes issues de
`store["catalogue_public"]`, moyenne les taux de fuite pour chaque `k`.

In [7]:
# Exercice 1 -- mesurer l'effet du top-k sur le taux de fuite moyen.
# Étape 1 : choisis plusieurs requêtes issues de catalogue_public.
# Étape 2 : pour k dans [5, 10, 20], calcule le taux de fuite moyen (sans filtre).
# resultats = {}  # TODO étudiant
pass

### Exercice 2 — Un reindexer qui refuse l'ambiguïté

`reindexer(..., environnement=None)` est la porte ouverte à l'accident.
Écris une variante `reindexer_sur` qui **refuse** d'écrire quand
l'environnement est `None`, au lieu d'écrire silencieusement dans
l'emplacement par défaut. Elle doit signaler le refus sans lever d'erreur
qui casserait l'exécution du notebook.

*Indice :* retourne une valeur sentinelle (par exemple la chaîne
`"REFUS : environnement non précisé"`) quand `environnement is None`,
au lieu de toucher au store.

In [8]:
# Exercice 2 -- écrire reindexer_sur(nouveau, store, env) qui refuse env=None.
# Contrat : si env is None, ne rien écrire et retourner un message explicite.
# def reindexer_sur(nouveau_corpus, store, environnement=None):
#     ...
# resultat = None  # TODO étudiant
pass

### Exercice 3 — Un test de non-régression pour l'écrasement

L'accident de réindexation est silencieux : rien ne l'annonce. Écris un
**test** (une assertion simple) qui détecte qu'un environnement a été
écrasé : il compare le compte de chunks d'un environnement avant et après
une opération, et signale toute chute. Le test doit **passer** quand
l'environnement est intact, et **échouer** (message clair) s'il a été
écrasé.

*Indice :* `assert len(store[env]) == compte_avant, f"{env} écrasé"`.

In [9]:
# Exercice 3 -- un test qui détecte l'écrasement silencieux d'un environnement.
# Étape 1 : mémorise le compte de chunks de 'archive_privee' avant l'opération.
# Étape 2 : définis assert_pas_ecrase(env, compte_avant) qui lève une AssertionError
#           claire si len(store[env]) != compte_avant.
# test = None  # TODO étudiant
pass

## Provenance et pour aller plus loin

Ce notebook matérialise le **Parcours 2** de `livresagites-parcours.md`
(RAG sur corpus et piège du multi-environnement). Il est le compagnon
exécutable de l'assertion « la séparation par environnement est un
contrôle d'accès ».

- **Parcours 3** (audit d'un serveur MCP) a son propre compagnon :
  `auditer-un-serveur-mcp.ipynb`.
- La fixture est **synthétique à 100 %** (numpy, seed=7) : aucun appel
  réseau, aucune clé, aucune donnée client. Les noms d'environnements
  (`catalogue_public`, `comite_lecture`…) sont des rôles d'accès
  génériques, pas des noms réels ; les chunks sont des vecteurs, pas de
  la prose.
- Le scénario de chevauchement (`catalogue_public` ↔ `comite_lecture`)
  est volontaire : il modélise le cas réaliste où deux environnements
  traitent du contenu sémantiquement proche — précisément celui où un
  filtre d'environnement devient indispensable.